### 1. 패키지 설치 + 환경변수 로드

In [1]:
%pip install -qU langchain langchain_openai langgraph

Note: you may need to restart the kernel to use updated packages.


In [2]:
from dotenv import load_dotenv
load_dotenv()

True

### 2. 그래프 준비

체크포인트 구조에 집중하기 위해 도구 없이 단일 노드 그래프로 축소함

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

class State(TypedDict):
    messages: Annotated[list, add_messages]

llm = ChatOpenAI(model="gpt-4o-mini")

def chatbot(state: State):
    return {"messages": [llm.invoke(state["messages"])]}

graph_builder = StateGraph(State)
graph_builder.add_node("chatbot", chatbot)
graph_builder.add_edge(START, "chatbot")
graph_builder.add_edge("chatbot", END)

memory = InMemorySaver()
graph = graph_builder.compile(checkpointer=memory)

### 3. 대화 두 번 진행

In [5]:
from langchain_core.runnables import RunnableConfig

config: RunnableConfig = {"configurable": {"thread_id": "1"}}

graph.invoke({"messages": [("user", "내 이름은 길동이야. 기억해줘")]}, config)
graph.invoke({"messages": [("user", "내 이름이 뭐라고 했지?")]}, config)

print("대화 완료")

대화 완료


### 4. StateSnapshot 구조

`get_state()` 가 돌려주는 값의 필드를 하나씩 확인함

| 필드 | 의미 |
|---|---|
| `values` | 이 체크포인트 시점의 State 값 |
| `next` | 다음에 실행할 노드 이름. 비어있으면 `()` → 실행 완료 |
| `config` | thread_id, checkpoint_ns, **checkpoint_id** |
| `metadata` | `source`(input/loop/update), `writes`(노드 출력), `step` |
| `created_at` | ISO 8601 생성 시각 |
| `parent_config` | 직전 체크포인트의 config. 최초는 `None` |
| `tasks` | 이 스텝에서 실행할 작업 |

In [6]:
snapshot = graph.get_state(config)

print("values 메시지 수:", len(snapshot.values["messages"]))
print("next:", snapshot.next)
print("config:", snapshot.config)
print("created_at:", snapshot.created_at)
print("parent_config:", snapshot.parent_config)
print("tasks:", snapshot.tasks)

values 메시지 수: 4
next: ()
config: {'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f18c19f-266f-62c4-8004-ba2bd0421e04'}}
created_at: 2026-07-30T13:24:09.410016+00:00
parent_config: {'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f18c19f-1a7f-6de4-8003-9d3a9e025443'}}
tasks: ()


### 5. metadata 확인

서드파티 유틸 없이 `json.dumps` 로 구조를 펼쳐 봄

In [7]:
import json

print(json.dumps(snapshot.metadata, ensure_ascii=False, indent=2, default=str))

{
  "source": "loop",
  "step": 4,
  "parents": {}
}


### 6. checkpoint_id

`config` 안의 `checkpoint_id` 가 이 시점을 가리키는 주소임. 05번 time travel에서 이 값을 사용함

In [8]:
parent_config = snapshot.parent_config or {}

print("현재 checkpoint_id:", snapshot.config.get("configurable", {}).get("checkpoint_id"))
print("직전 checkpoint_id:", parent_config.get("configurable", {}).get("checkpoint_id"))

현재 checkpoint_id: 1f18c19f-266f-62c4-8004-ba2bd0421e04
직전 checkpoint_id: 1f18c19f-1a7f-6de4-8003-9d3a9e025443


### 7. get_state_history — 스텝별 체크포인트 나열

최신 것이 먼저 나옴

In [9]:
history = list(graph.get_state_history(config))

print(f"체크포인트 {len(history)}개\n")
for state in history:
    metadata = state.metadata or {}
    checkpoint_id = state.config.get("configurable", {}).get("checkpoint_id")
    print(f"step={metadata.get('step', ''):>2} "
          f"source={metadata.get('source', ''):<6} "
          f"next={str(state.next):<12} "
          f"messages={len(state.values.get('messages', []))} "
          f"id={checkpoint_id}")

체크포인트 6개

step= 4 source=loop   next=()           messages=4 id=1f18c19f-266f-62c4-8004-ba2bd0421e04
step= 3 source=loop   next=('chatbot',) messages=3 id=1f18c19f-1a7f-6de4-8003-9d3a9e025443
step= 2 source=input  next=('__start__',) messages=2 id=1f18c19f-1a7a-6fc0-8002-05a32dfa040c
step= 1 source=loop   next=()           messages=2 id=1f18c19f-1a78-68da-8001-9b66605c17ef
step= 0 source=loop   next=('chatbot',) messages=1 id=1f18c19f-053a-68dc-8000-2a997857270c
step=-1 source=input  next=('__start__',) messages=0 id=1f18c19f-0535-6aab-bfff-21e23549b196


### 8. 정리

- 체크포인트는 **스텝마다** 쌓이며, `get_state()` 는 그중 가장 최신 것을 돌려줌
- `checkpoint_id` 로 특정 시점을 지목할 수 있음 → 05번 time travel의 전제
- `metadata.writes` 에는 그 스텝에서 각 노드가 무엇을 반환했는지가 담김
- 참고: [Checkpointers](https://docs.langchain.com/oss/python/langgraph/checkpointers)